# Recomendación colaborativa de películas: ejecución secuencial

Ejecuta primero 00. Guarda y cierra 01–05. Este maestro los ejecuta en orden, con kernels independientes del mismo Python activo. Se detiene ante errores y reemplaza los reportes de las etapas completadas. No ejecuta otros proyectos ni cambia SmartFood o churn.

In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import Markdown, display
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("movielens_recommender")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre el notebook desde el proyecto, notebooks o su carpeta contenedora.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
for folder in ("data/raw", "data/intermediate", "data/processed", "data/output", "models", "reports"):
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
from movielens_recommender.alerts import configurar_notebook, mostrar_alertas
configurar_notebook()

TIMEOUT_POR_CELDA = 900


In [ ]:
from movielens_recommender.execution import ejecutar_secuencia
if ejecutar_secuencia(PROJECT_ROOT, timeout=TIMEOUT_POR_CELDA):
    raise RuntimeError("Secuencia detenida. Revisa reports/ejecucion/resumen.json.")


In [ ]:
from movielens_recommender.common import read_json
import pandas as pd
display(pd.DataFrame(read_json(PROJECT_ROOT / "reports/ejecucion/resumen.json")["notebooks"]))
mostrar_alertas(PROJECT_ROOT / "reports/alertas/06_ejecucion_completa.json")
from movielens_recommender.informe import generar_informe
from IPython.display import Markdown
informe = generar_informe(PROJECT_ROOT)
display(Markdown("[Abrir informe integral](../reports/informe_final/informe_final.md)"))
